In [ ]:
# Testing qDRIFT for several hydrogen chains

from pyscf import gto, scf
import numpy as np
import openfermion as of
from qarp.operators.compat import from_openfermion
from qarp.operators.functions import count_qubits

from qarp.operators import JordanWigner
from qarp.operators.pyscf import fermion_operator_from_mf, onv_from_mf
from qarp.blocks import ComputationalBasisStateBlock, UCCBlock, CompositeBlock
from qarp.algorithms import TermwiseHadamardTest, StateVector
from qarp.engines import QarpEngine
from qarp.operators import qDRIFT
import matplotlib.pylab as plt


# This for H2
bl = 0.735
geometry = f"H 0 0 0; H 0 0 {bl}"
mol = gto.M(atom=geometry, basis="sto3g", verbose=-1, symmetry=True)
mol.build()
mf = scf.RHF(mol)
mf.kernel()
h2_mf = mf
####


# This for LiH 
mol = gto.M(atom="H 0 0 0; Li 0 0 1.59", basis="sto3g", symmetry=True, verbose=-1)
mol.build()
mf = scf.RHF(mol)
mf.kernel()
lih_mf = mf
####

# This for H3
bl = 0.735
geometry = f"H 0 0 0; H 0 0 {bl}; H 0 0 {2*bl}"
mol = gto.M(atom=geometry, basis="sto3g", verbose=-1, symmetry=True, spin=1)
mol.build()
mf = scf.RHF(mol)
mf.kernel()
h3_mf = mf
####

# This for H4
bl = 0.735
geometry = f"H 0 0 0; H 0 0 {bl}; H 0 0 {bl * 2}; H 0 0 {bl * 3}"
mol = gto.M(atom=geometry, basis="sto3g", verbose=-1, symmetry=True)
mol.build()
mf = scf.RHF(mol)
mf.kernel()
h4_mf = mf
####

# This for a random Hamiltonian 
H_random = of.FermionOperator()
for i in range(4):
    H_random += of.FermionOperator(f'{i}^ {i}', np.random.rand()) 
for i, j in [(0, 1), (1, 2), (2, 3), (3, 0)]:
    H_random += of.FermionOperator(f'{i}^ {j}', np.random.rand())
    H_random += of.FermionOperator(f'{j}^ {i}', np.random.rand())
for i, j in [(0, 2), (1, 3)]:
    H_random += of.FermionOperator(f'{i}^ {i} {j}^ {j}', np.random.rand())
H_random += of.hermitian_conjugated(H_random)
H_random = of.normal_ordered(H_random)
H_random = from_openfermion(H_random)  # native boundary for JW encode
####


mf = h2_mf  # change here the molecule you want to apply BRG

fermion_operator = fermion_operator_from_mf(mf)
qop = JordanWigner().encode_operator(fermion_operator)

onv = onv_from_mf(mf)

ref = ComputationalBasisStateBlock(onv)
ucc = UCCBlock(onv, singles=True, doubles=True)
ansatz = CompositeBlock([ref, ucc])
ansatz.build()
symbol_map = dict(zip(ansatz.symbols, [.5]*len(ansatz.symbols)))

modes = count_qubits(qop)

print("How many terms in chosen QubitOperator? ", len(qop.terms))


In [ ]:
# Measure whole Hamiltonian with Statevector and TermwiseHadamardTest approaches 

meas_sampled = TermwiseHadamardTest(bra=ansatz, ket=ansatz, operator=qop, n_shots=None)
meas_SV = StateVector(bra=ansatz, ket=ansatz, operator=qop)

my_engine = QarpEngine()
my_engine.build([meas_sampled, meas_SV])

results = my_engine.run(symbol_map)

print("THT and SV results for chosen QO (should be the same): ", results ) 


In [ ]:
# Use qDRIFT to obtain effective Hamiltonians with a fixed number of terms by sampling it, and measure it

results_qdrift = []
for i in list(range(10,80,10)):
    effH = qDRIFT(qop, samples=i).qdrift()
    meas_SV = StateVector(bra=ansatz, ket=ansatz, operator=effH)
    
    my_engine = QarpEngine()
    my_engine.build([meas_SV])

    results_qdrift.append( my_engine.run(symbol_map) )


In [ ]:
# Compute the absolute differences with full Hamiltonian
abs_diff = [np.abs(results_qdrift[i][0] - results[1]) for i in range(len(results_qdrift))]

In [ ]:
plt.plot(list(range(10,80,10)), abs_diff, linestyle="-.", linewidth=2, label="qDRIFT absolute diffs", color="black")



plt.ylabel("diff (a.u.)", fontsize=20)
plt.xlabel("Samples", fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.legend(fontsize=16)


In [ ]:
# Plot of absolute value of coefficients of the QubitOperator, where we can observe there are a few terms more important in magnitude than others

plt.plot(sorted(np.abs(list(qop.terms.values())), reverse=True), lw = 2, color="black")


plt.ylabel("abs(coef)", fontsize=20)
plt.xlabel("#", fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)


In [ ]:
# A refined version of qDRIFT can be made by fixing the most-likely terms and sampling the remaining ones: partially-randomized qDRIFT 

results_qdrift = []
for i in list(range(10,80,10)):
    effH = qDRIFT(qop, samples=i, ratio=0.3).partially_randomized() # 30% of all the terms are accounted for
    meas_SV = StateVector(bra=ansatz, ket=ansatz, operator=effH)
    
    my_engine = QarpEngine()
    my_engine.build([meas_SV])

    results_qdrift.append( my_engine.run(symbol_map) )

In [ ]:
# Compute the absolute differences with full Hamiltonian
abs_diff = [np.abs(results_qdrift[i][0] - results[1]) for i in range(len(results_qdrift))]

In [ ]:
# Same plot, we observe that now errors are smaller in general

plt.plot(list(range(10,80,10)), abs_diff, linestyle="-.", linewidth=2, label="qDRIFT absolute diffs", color="black")


plt.ylabel("diff (a.u.)", fontsize=20)
plt.xlabel("Samples", fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.legend(fontsize=16)